# OmniVoice Project Studio — Kaggle Local SSD

This notebook intentionally uses **Kaggle local execution storage only**.

```text
/kaggle/working/OmniVoiceStudio
├── projects/
├── voices/
├── project-queue.json
└── hardware-quality.json
```

No Google Drive, rclone, or remote persistence is configured in this phase. The workspace is fast but ephemeral: when the Kaggle session is discarded, local files disappear unless you export them separately. `/kaggle/input` is read-only source material, not a Project Studio workspace.


In [ ]:
# Install the Kaggle-local Project Studio branch. Kaggle Internet must be enabled
# for GitHub/Hugging Face downloads on a fresh runtime.
!pip install -q --upgrade "git+https://github.com/binhminhanh1235/OmniVoice.git@feat/kaggle-local-workspace"


In [ ]:
from pathlib import Path
import shutil
import torch

from omnivoice.runtime_workspace import detect_runtime_workspace, ensure_runtime_workspace
from omnivoice.hardware_quality import detect_hardware

runtime = ensure_runtime_workspace(detect_runtime_workspace())
hardware = detect_hardware()

print("Runtime:", runtime.summary())
print("Hardware:", hardware.summary())
print("Workspace:", runtime.root)
print("Input source:", runtime.input_root)
usage = shutil.disk_usage(Path("/kaggle/working"))
print(f"Local disk free: {usage.free / 1024**3:.1f} GB")

if runtime.environment != "kaggle":
    raise RuntimeError(f"Expected Kaggle runtime, detected: {runtime.environment}")
if not torch.cuda.is_available():
    raise RuntimeError("Enable a GPU accelerator in Kaggle Notebook settings.")


## Launch Project Studio

Recommended Kaggle flow:

1. **Text Doctor** — clean the script.
2. **Voice Doctor** — create/reuse a local voice reference.
3. **Project Studio** — create and preview projects.
4. **Project Queue** — queue many local projects.
5. **Hardware & Quality** — choose SAFE / BALANCED / FAST.

All project/chunk/section status files and generated WAVs stay under `/kaggle/working/OmniVoiceStudio`.


In [ ]:
!omnivoice-project-studio \
  --model k2-fsa/OmniVoice \
  --workspace /kaggle/working/OmniVoiceStudio \
  --asr-model openai/whisper-small.en \
  --asr-device cpu \
  --share


## Local workspace reminder

This phase deliberately has **no remote persistence**. `section-status.json` and `project-queue.json` make generation crash-resumable only while the local workspace still exists. Google Drive synchronization will be designed as a separate persistence layer later.
